# Multi-agent Matching Experiment - 4 Agents

4エージェントのシーンで、全6 pair の RoCo / Pairwise RRWM / Partial OT と、4通りの3-wise RRWM triple を比較します。既存の k-wise 実装は3-wiseなので、4台版では各3台組み合わせごとに実行します。

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path):
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "src" / "simulation.py").exists():
            return path
    return None


repo_dir = find_repo_root(Path.cwd())
if repo_dir is None:
    raise FileNotFoundError("Could not find repository root. Run this notebook inside the repository.")

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", Path.cwd())

In [ ]:
from datetime import datetime
from itertools import combinations
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from src.simulation import count_detection_labels, generate_detections, generate_true_scene
from src.visualization import plot_detected_scene, plot_pairwise_matching_result, plot_true_scene
from src.graph_matching import (
    KWISE_PARAMS,
    PARTIAL_OT_PARAMS,
    ROCO_PARAMS,
    RRWM_PARAMS,
    evaluate_kwise_matching_3agents,
    evaluate_pairwise_matching,
    kwise_rrwm_matching_3agents,
    kwise_to_pairwise_matches,
    run_consistent_kwise_rrwm_matching,
    run_pairwise_rrwm_all_pairs,
    run_roco_all_pairs,
)
from src.roco_iterative import (
    ROCO_ITERATIVE_PARAMS,
    build_agent_pose_error_dataframe,
    build_iterative_matching_dataframe,
    build_object_pose_error_dataframe,
    build_pose_error_history_dataframe,
    build_pose_error_summary_dataframe,
    evaluate_iterative_roco_pose_error_history,
    evaluate_iterative_roco_pose_errors,
    run_iterative_pairwise_rrwm_pose_adjustment,
    run_iterative_partial_ot_pose_adjustment,
    run_iterative_roco_pose_adjustment,
    run_partial_ot_matching_all_pairs,
)

num_agents = 4
agent_ids = tuple(range(num_agents))
agent_pairs = list(combinations(agent_ids, 2))
agent_triples = list(combinations(agent_ids, 3))

print("agent_pairs:", agent_pairs)
print("agent_triples:", agent_triples)

In [ ]:
experiment_seed = 7
rng = np.random.default_rng(experiment_seed)

roco_params = ROCO_PARAMS.copy()
roco_params.update({"tau2": 6.0, "tau1": 0.3, "lambda_dist": 1.0, "neighbor_radius": 15.0})

rrwm_params = RRWM_PARAMS.copy()
rrwm_params.update(
    {
        "candidate_radius": 6.0,
        "unary_weight": 2.0,
        "pairwise_weight": 1.0,
        "sigma_pos": 2.0,
        "sigma_yaw_deg": 15.0,
        "sigma_edge": 1.5,
        "max_iter": 10000,
        "alpha": 0.2,
        "beta": 30.0,
        "score_threshold": 0.02,
    }
)

kwise_params = KWISE_PARAMS.copy()
kwise_params.update(rrwm_params)
kwise_params.update({"max_iter": 1000})

partial_ot_params = PARTIAL_OT_PARAMS.copy()
partial_ot_params.update(
    {
        "epsilon": 8.0,
        "epsilon_decay": 0.93,
        "beta2": 0.8,
        "outer_iter": 30,
        "inner_iter": 100,
        "tol": 1e-5,
        "match_score_threshold": 1e-4,
    }
)

iterative_params = ROCO_ITERATIVE_PARAMS.copy()
iterative_params.update({"max_iter": 10, "damping": 0.8, "pose_tol": 1e-3, "min_matches_for_pose": 2})

timestamp = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d_%H%M%S_JST")
output_dir = Path("results") / f"{timestamp}_4agents_seed_{experiment_seed}"
output_dir.mkdir(parents=True, exist_ok=True)
print("output_dir:", output_dir)

In [ ]:
agents_gt, objects_gt = generate_true_scene(
    num_agents=num_agents,
    num_objects=16,
    seed=experiment_seed,
    rng=rng,
)
agent_pose_est, detections_by_agent = generate_detections(
    agents_gt,
    objects_gt,
    sensing_range=34.0,
    fov_deg=160.0,
    detection_prob=0.9,
    object_pos_noise_std=0.7,
    object_yaw_noise_std_deg=4.0,
    agent_pos_noise_std=1.0,
    agent_yaw_noise_std_deg=3.0,
    num_outliers_per_agent=2,
    seed=experiment_seed,
    rng=rng,
)

print("detection counts:", count_detection_labels(detections_by_agent))
plot_true_scene(agents_gt, objects_gt, save_path=output_dir / f"true_scene_4agents_seed_{experiment_seed}.png")
plot_detected_scene(agents_gt, agent_pose_est, detections_by_agent, save_path=output_dir / f"observed_scene_4agents_seed_{experiment_seed}.png")

In [ ]:
def pairwise_records(method_name, results):
    records = []
    for i, j in agent_pairs:
        eval_result = results[(i, j)]["evaluation"]
        tp = int(eval_result["correct"])
        predicted = int(eval_result["predicted_matches"])
        gt = int(eval_result["gt_matches"])
        records.append(
            {
                "method": method_name,
                "pair": f"{i}-{j}",
                "tp": tp,
                "fp": predicted - tp,
                "fn": gt - tp,
                "gt_matches": gt,
                "predicted_matches": predicted,
                "precision": float(eval_result["precision"]),
                "recall": float(eval_result["recall"]),
            }
        )
    return records


def local_pair_for_triple(triple, pair):
    return (triple.index(pair[0]), triple.index(pair[1]))


def kwise_pairwise_matches_for_global_pair(kwise_matches, triple, pair):
    return kwise_to_pairwise_matches(kwise_matches, pair=local_pair_for_triple(triple, pair))


def evaluate_triple_as_pairwise(triple, kwise_matches):
    records = []
    for pair in combinations(triple, 2):
        matches = kwise_pairwise_matches_for_global_pair(kwise_matches, triple, pair)
        eval_result = evaluate_pairwise_matching(
            detections_by_agent[pair[0]],
            detections_by_agent[pair[1]],
            matches,
        )
        tp = int(eval_result["correct"])
        predicted = int(eval_result["predicted_matches"])
        gt = int(eval_result["gt_matches"])
        records.append(
            {
                "method": f"3-wise RRWM {triple}",
                "triple": "-".join(map(str, triple)),
                "pair": f"{pair[0]}-{pair[1]}",
                "tp": tp,
                "fp": predicted - tp,
                "fn": gt - tp,
                "gt_matches": gt,
                "predicted_matches": predicted,
                "precision": float(eval_result["precision"]),
                "recall": float(eval_result["recall"]),
            }
        )
    return records


In [ ]:
roco_results = run_roco_all_pairs(detections_by_agent, agent_pairs, params=roco_params)
rrwm_results = run_pairwise_rrwm_all_pairs(detections_by_agent, agent_pairs, params=rrwm_params)

partial_ot_matches, partial_ot_extras = run_partial_ot_matching_all_pairs(
    detections_by_agent,
    agent_pairs,
    partial_ot_params=partial_ot_params,
)
partial_ot_results = {
    pair: {
        "matches": matches,
        "evaluation": evaluate_pairwise_matching(detections_by_agent[pair[0]], detections_by_agent[pair[1]], matches),
    }
    for pair, matches in partial_ot_matches.items()
}
consistent_kwise_matches, consistent_kwise_extras = run_consistent_kwise_rrwm_matching(
    detections_by_agent,
    agent_ids=agent_ids,
    kwise_params=kwise_params,
    reference_agent_id=0,
    score_threshold=kwise_params["score_threshold"],
)
consistent_kwise_results = {
    pair: {
        "matches": matches,
        "evaluation": evaluate_pairwise_matching(detections_by_agent[pair[0]], detections_by_agent[pair[1]], matches),
    }
    for pair, matches in consistent_kwise_matches.items()
}

kwise_triple_results = {}
kwise_triple_records = []
kwise_pairwise_records = []
for triple in agent_triples:
    result = consistent_kwise_extras["kwise_results_by_triple"][triple]
    matches = result["matches"]
    evaluation = result["evaluation"]
    kwise_triple_results[triple] = result
    kwise_triple_records.append({"triple": "-".join(map(str, triple)), **evaluation})
    kwise_pairwise_records.extend(evaluate_triple_as_pairwise(triple, matches))

comparison_records = []
comparison_records.extend(pairwise_records("RoCo-style", roco_results))
comparison_records.extend(pairwise_records("Pairwise RRWM", rrwm_results))
comparison_records.extend(pairwise_records("Partial OT", partial_ot_results))
comparison_records.extend(pairwise_records("Consistent 3-wise RRWM", consistent_kwise_results))
comparison_df = pd.DataFrame.from_records(comparison_records)
kwise_triple_df = pd.DataFrame.from_records(kwise_triple_records)
kwise_pairwise_df = pd.DataFrame.from_records(kwise_pairwise_records)

comparison_df.to_csv(output_dir / "matching_comparison_4agents_pairwise_methods.csv", index=False)
kwise_triple_df.to_csv(output_dir / "kwise_triple_comparison_4agents.csv", index=False)
kwise_pairwise_df.to_csv(output_dir / "kwise_pairwise_decomposed_4agents.csv", index=False)

display(comparison_df)
display(kwise_triple_df)
display(kwise_pairwise_df)

In [ ]:
iterative_roco_result = run_iterative_roco_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    roco_params=roco_params,
    anchor_agent_id=0,
    **iterative_params,
)
iterative_pairwise_rrwm_result = run_iterative_pairwise_rrwm_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    rrwm_params=rrwm_params,
    anchor_agent_id=0,
    **iterative_params,
)
iterative_partial_ot_result = run_iterative_partial_ot_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    partial_ot_params=partial_ot_params,
    anchor_agent_id=0,
    **iterative_params,
)

iterative_results = {
    "RoCo": iterative_roco_result,
    "Pairwise RRWM": iterative_pairwise_rrwm_result,
    "Partial OT": iterative_partial_ot_result,
}

iterative_matching_df = pd.concat(
    [build_iterative_matching_dataframe(result) for result in iterative_results.values()],
    ignore_index=True,
)
pose_error_results = {name: evaluate_iterative_roco_pose_errors(result, agents_gt, objects_gt) for name, result in iterative_results.items()}
pose_error_summary_df = pd.concat(
    [build_pose_error_summary_dataframe(result).assign(method=name) for name, result in pose_error_results.items()],
    ignore_index=True,
)
pose_error_history_df = pd.concat(
    [build_pose_error_history_dataframe(evaluate_iterative_roco_pose_error_history(result, agents_gt, objects_gt)) for result in iterative_results.values()],
    ignore_index=True,
)

iterative_matching_df.to_csv(output_dir / "iterative_matching_history_4agents.csv", index=False)
pose_error_summary_df.to_csv(output_dir / "pose_error_summary_4agents.csv", index=False)
pose_error_history_df.to_csv(output_dir / "pose_error_history_4agents.csv", index=False)

for method_name, pose_error_result in pose_error_results.items():
    label = method_name.lower().replace(" ", "_")
    build_agent_pose_error_dataframe(pose_error_result).to_csv(output_dir / f"agent_pose_errors_{label}_4agents.csv", index=False)
    build_object_pose_error_dataframe(pose_error_result).to_csv(output_dir / f"object_pose_errors_{label}_4agents.csv", index=False)

display(iterative_matching_df)
display(pose_error_summary_df)
display(pose_error_history_df)

In [ ]:
plot_labels = {
    "RoCo": "Iterative_RoCo_4agents",
    "Pairwise RRWM": "Iterative_Pairwise_RRWM_4agents",
    "Partial OT": "Iterative_Partial_OT_4agents",
}

for method_name, result in iterative_results.items():
    plot_detected_scene(
        agents_gt,
        result["agent_poses"],
        result["detections_by_agent"],
        save_path=output_dir / f"{plot_labels[method_name]}_adjusted_scene_seed_{experiment_seed}.png",
    )

    for i, j in agent_pairs:
        plot_pairwise_matching_result(
            result["detections_by_agent"][i],
            result["detections_by_agent"][j],
            result["pairwise_matches"][(i, j)],
            agent_i=i,
            agent_j=j,
            title=f"{method_name} iterative result: Agent {i} - Agent {j}",
            params=iterative_params,
            objects_gt=objects_gt,
            save_path=output_dir / f"{plot_labels[method_name]}_{i}{j}_seed_{experiment_seed}.png",
        )

print("saved to:", output_dir)

In [ ]:
from html import escape


def build_experiment_report(output_dir: Path):
    output_dir = Path(output_dir)
    csv_paths = sorted(output_dir.glob("*.csv"))
    image_paths = sorted(output_dir.glob("*.png"))

    csv_sections = []
    for csv_path in csv_paths:
        df = pd.read_csv(csv_path)
        csv_sections.append(
            f"""
            <section>
              <div class=\"section-header\"><h2>{escape(csv_path.stem)}</h2><a href=\"{escape(csv_path.name)}\">CSV</a></div>
              <div class=\"table-wrap\">{df.to_html(index=False, classes='data-table', border=0)}</div>
            </section>
            """
        )

    image_cards = [
        f"""
        <a class=\"image-card\" href=\"{escape(path.name)}\">
          <img src=\"{escape(path.name)}\" alt=\"{escape(path.stem)}\">
          <span>{escape(path.stem)}</span>
        </a>
        """
        for path in image_paths
    ]

    html = f"""
    <!doctype html>
    <html lang=\"ja\">
    <head>
      <meta charset=\"utf-8\">
      <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
      <title>4-Agent Experiment Summary - {escape(output_dir.name)}</title>
      <style>
        body {{ margin: 0; color: #202124; background: #f7f7f4; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
        main {{ max-width: 1200px; margin: 0 auto; padding: 32px 20px 56px; }}
        h1 {{ margin: 0 0 6px; font-size: 28px; }}
        h2 {{ margin: 0; font-size: 18px; }}
        .meta {{ margin: 0 0 28px; color: #5f6368; }}
        section {{ margin-top: 28px; }}
        .section-header {{ display: flex; align-items: center; justify-content: space-between; gap: 16px; margin-bottom: 10px; }}
        .section-header a {{ color: #0b57d0; text-decoration: none; font-size: 14px; }}
        .image-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(260px, 1fr)); gap: 14px; }}
        .image-card {{ display: block; overflow: hidden; border: 1px solid #dadce0; border-radius: 8px; background: white; color: inherit; text-decoration: none; }}
        .image-card img {{ display: block; width: 100%; height: 190px; object-fit: contain; background: #fff; border-bottom: 1px solid #eceff1; }}
        .image-card span {{ display: block; padding: 9px 10px; font-size: 13px; color: #3c4043; overflow-wrap: anywhere; }}
        .table-wrap {{ overflow-x: auto; border: 1px solid #dadce0; border-radius: 8px; background: white; }}
        table.data-table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
        .data-table th, .data-table td {{ padding: 7px 9px; border-bottom: 1px solid #eceff1; text-align: right; white-space: nowrap; }}
        .data-table th {{ background: #eef2f7; color: #202124; font-weight: 650; }}
        .data-table th:first-child, .data-table td:first-child {{ text-align: left; }}
      </style>
    </head>
    <body><main>
      <h1>4-Agent Experiment Summary</h1>
      <p class=\"meta\">{escape(output_dir.name)}</p>
      <section><div class=\"section-header\"><h2>Figures</h2></div><div class=\"image-grid\">{''.join(image_cards)}</div></section>
      {''.join(csv_sections)}
    </main></body></html>
    """
    report_path = output_dir / "summary.html"
    report_path.write_text(html, encoding="utf-8")
    return report_path


report_path = build_experiment_report(output_dir)
print("report_path:", report_path)